In [ ]:
!pip install transformers[torch] datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Load your specific file
df = pd.read_csv('dark-patterns-v2.csv')

# 2. Cleaning (Matches the processing we just did)
df = df.dropna(subset=['Pattern String'])
unique_labels = df['Pattern Category'].unique().tolist()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
df['label'] = df['Pattern Category'].map(label2id)
df['text'] = df['Pattern String'].astype(str)

# 3. Prepare Dataset
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
dataset = Dataset.from_pandas(df[['text', 'label']])

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_ds = dataset.map(tokenize_func, batched=True).train_test_split(test_size=0.2)

# 4. Model Setup
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

# 5. Training (Run for 3 epochs for the progress report results)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_steps=10,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
)

trainer.train()

Map:   0%|          | 0/1512 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.180451,0.169139
2,0.057420,0.135806
3,0.033245,0.121418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=228, training_loss=0.23708824878721907, metrics={'train_runtime': 190.7633, 'train_samples_per_second': 19.013, 'train_steps_per_second': 1.195, 'total_flos': 480502096468992.0, 'train_loss': 0.23708824878721907, 'epoch': 3.0})